# Third-Party Vendor Governance: Vendor Risk Scoring

## InsureLogic AI: Vendor Risk Assessment & SLA Compliance Dashboard

This notebook implements vendor risk scoring and SLA compliance monitoring for InsureLogic AI's 5 AI vendor relationships.

---

### Quick Reference: Key Concepts

**Vendor Risk Scoring Criteria**:
- **Security Posture** (30%): Certifications (SOC2, ISO27001), encryption, access controls
- **Data Handling** (25%): Data residency, retention policies, processing agreements
- **Transparency** (20%): Model explainability, bias testing, documentation
- **Financial Stability** (15%): Revenue, market position, business continuity
- **SLA Quality** (10%): Uptime guarantees, response times, penalty structures

**Risk Score Scale**: 1 (Lowest Risk) → 5 (Highest Risk)
- 1.0-1.5: Highly Recommended — Strong across all dimensions
- 1.5-2.5: Recommended — Good overall with minor gaps
- 2.5-3.5: Acceptable with Monitoring — Notable gaps requiring attention
- 3.5-5.0: Not Recommended — Significant governance concerns

**Vendor Governance Best Practices**:
- Maintain 2+ active vendor relationships for redundancy
- Review vendor risk quarterly; SLA compliance monthly
- Test failover procedures at least annually
- Require 7+ days advance notice for model updates

### Scenario
InsureLogic AI uses GenAI for claims processing, fraud detection, and customer chatbots across 5 vendors. After a vendor model update caused false claim denials, the CRO wants formal risk scoring and SLA monitoring.

## Step 1: Setup and Vendor Data

In [ ]:
# ============================================================
# VENDOR EVALUATION SCORES (0-100 per criterion)
# ============================================================
import pandas as pd
vendors_df = pd.DataFrame({
    'Vendor': ['NovaMind API', 'CloudVault AI', 'TitanScale AI', 'DeepLens AI', 'SafeGuard AI'],
    'Security_Score': [85, 95, 95, 15, 85],
    'Data_Handling_Score': [75, 90, 85, 10, 85],
    'Model_Transparency': [90, 70, 70, 40, 90],
    'Financial_Stability': [95, 100, 100, 85, 85],
    'SLA_Compliance': [85, 90, 95, 60, 85],
    'Exit_Strategy': [90, 95, 90, 50, 95],
    'AI_Model_Governance': [70, 75, 80, 20, 85]
})

In [ ]:
# ============================================================
# SLA PERFORMANCE DATA — 30-day tracking for each vendor
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
from math import pi

np.random.seed(42)

# Generate 30 days of SLA data for each vendor
vendors = ['NovaMind API', 'CloudVault AI', 'TitanScale AI', 'DeepLens AI', 'SafeGuard AI']
days = list(range(1, 31))

sla_records = []
for vendor in vendors:
    # Set baseline performance characteristics per vendor
    if vendor == 'NovaMind API':
        uptime_mean, uptime_std = 99.92, 0.03
        latency_mean, latency_std = 450, 50
    elif vendor == 'CloudVault AI':
        uptime_mean, uptime_std = 99.98, 0.01
        latency_mean, latency_std = 380, 30
    elif vendor == 'TitanScale AI':
        uptime_mean, uptime_std = 99.99, 0.005
        latency_mean, latency_std = 320, 25
    elif vendor == 'DeepLens AI':
        uptime_mean, uptime_std = 99.75, 0.15  # Poor performer
        latency_mean, latency_std = 950, 200
    else:  # SafeGuard AI
        uptime_mean, uptime_std = 99.95, 0.02
        latency_mean, latency_std = 420, 40
    
    for day in days:
        uptime = min(100, max(98, np.random.normal(uptime_mean, uptime_std)))
        latency = max(100, np.random.normal(latency_mean, latency_std))
        sla_records.append({
            'Vendor': vendor,
            'Day': day,
            'Uptime': round(uptime, 3),
            'Latency_ms': round(latency, 1)
        })

sla_df = pd.DataFrame(sla_records)

print(f"SLA performance data: {len(sla_df)} records across {len(vendors)} vendors")
print(f"\nSample data (first 5 rows):")
print(sla_df.head().to_string(index=False))

## Step 2: Define Risk Scoring Weights

Vendor risk scoring requires weighting criteria by importance. This is a governance decision — the weights reflect what matters most for InsureLogic's insurance context.

**Your task**: Define the scoring weights dictionary. Consider that for an insurance company:
- Security is paramount (customer financial data)
- Data handling matters for regulatory compliance
- Transparency is critical for claims explainability
- Financial stability ensures vendor longevity
- SLA quality affects operational reliability

In [ ]:
scoring_weights = {
    'Security_Score': 0.27,
    'Data_Handling_Score': 0.18,
    'Model_Transparency': 0.18,
    'Financial_Stability': 0.14,
    'SLA_Compliance': 0.09,
    'Exit_Strategy': 0.04,
    'AI_Model_Governance': 0.10
}

total = sum(scoring_weights.values())
print(f'Total weight: {total:.2f}')
print('\nWeight allocation rationale:')
print('  Security (27%): Highest — insurance handles sensitive financial/health data')
print('  Data Handling (18%): High — regulatory compliance for policyholder data')
print('  Model Transparency (18%): High — regulators may require claims decision explainability')
print('  Financial Stability (14%): Medium — vendor longevity matters for multi-year contracts')
print('  SLA Compliance (9%): Lower — important but measurable and enforceable via contracts')
print('  Exit Strategy (4%): Lowest — important but a one-time consideration')
print('  AI Model Governance (10%): Critical — GenAI-specific: training data, updates, fine-tuning portability, liability')

## Step 3: Calculate Risk Scores

Convert vendor evaluation scores into a single risk score (1-5 scale) using your weights.

**Your task**: Implement the weighted score calculation. The logic:
1. For each vendor, compute a weighted average of their scores (0-100 scale)
2. Convert to risk scale: 100 → risk 1.0 (lowest), 0 → risk 5.0 (highest)
   Formula: `risk = 5 - (weighted_score / 100) * 4`

In [ ]:
def calculate_risk_scores(vendor_data, weights):
    """Calculate weighted risk scores for each vendor. Returns list of scores (1-5)."""
    risk_scores = []
    for _, row in vendor_data.iterrows():
        weighted_score = sum(row[metric] * weight for metric, weight in weights.items())
        risk_level = 5 - (weighted_score / 100) * 4
        risk_scores.append(round(risk_level, 2))
    return risk_scores

vendors_df['Risk_Score'] = calculate_risk_scores(vendors_df, scoring_weights)
print('Vendor Risk Scores:')
print(vendors_df[['Vendor', 'Risk_Score']].to_string(index=False))

## Step 4: Vendor Recommendation Logic

Based on risk scores and SLA compliance, assign a recommendation category for each vendor.

**Your task**: Implement the recommendation function using these governance thresholds:
- Risk ≤ 1.5 AND SLA compliant → "Highly Recommended"
- Risk ≤ 2.5 AND SLA compliant → "Recommended"
- Risk ≤ 3.5 → "Acceptable (with monitoring)"
- Risk > 3.5 → "Not Recommended"

In [ ]:
# First, calculate SLA compliance (pre-provided)
uptime_threshold = 99.9
latency_threshold = 800

compliance_df = sla_df.groupby('Vendor').agg(
    Avg_Uptime=('Uptime', 'mean'),
    Avg_Latency=('Latency_ms', 'mean')
).round(3).reset_index()

compliance_df['SLA_Compliant'] = (
    (compliance_df['Avg_Uptime'] >= uptime_threshold) & 
    (compliance_df['Avg_Latency'] <= latency_threshold)
)

# Merge risk scores with compliance
summary = vendors_df[['Vendor', 'Risk_Score']].merge(compliance_df, on='Vendor')

def get_recommendation(risk_score, sla_compliant):
    """Return recommendation string based on risk score and SLA compliance."""
    if risk_score <= 1.5 and sla_compliant:
        return 'Highly Recommended'
    elif risk_score <= 2.5 and sla_compliant:
        return 'Recommended'
    elif risk_score <= 3.5:
        return 'Acceptable (with monitoring)'
    else:
        return 'Not Recommended'

summary['Recommendation'] = summary.apply(
    lambda row: get_recommendation(row['Risk_Score'], row['SLA_Compliant']), axis=1
)

print('Vendor Assessment Summary:')
print(summary.to_string(index=False))

## Step 4.5: Scenario Analysis — Model Update Incident

The InsureLogic incident mentioned in the scenario involved an unannounced vendor model update that caused false claim denials.

**Critical Question**: Which vendor(s) in your assessment have the highest risk around model update notification and change management?

In [ ]:
# Analyze model update notification risk
print("\n" + "="*80)
print("MODEL UPDATE NOTIFICATION SLA RISK ANALYSIS")
print("="*80)

# This criterion is part of AI_Model_Governance
# Vendors with low AI_Model_Governance scores have poor update notification practices
model_gov_scores = vendors_df[['Vendor', 'AI_Model_Governance']].sort_values('AI_Model_Governance')

print("\nAI Model Governance Scores (includes Model Update Notification):")
print("Lower scores = higher risk of unannounced updates")
print(model_gov_scores.to_string(index=False))

high_risk_vendors = model_gov_scores[model_gov_scores['AI_Model_Governance'] < 50]['Vendor'].tolist()
if high_risk_vendors:
    print(f"\nHIGH RISK for unannounced updates: {', '.join(high_risk_vendors)}")
    print("\nMitigation:")
    print("  - Require 14-day advance notice clauses in contracts")
    print("  - Implement staged rollout: test on shadow traffic before production")
    print("  - Monitor model outputs continuously for performance degradation")
else:
    print("\nNo high-risk vendors identified for model update notification.")

## Step 5: Risk Comparison Chart

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
sorted_v = vendors_df.sort_values('Risk_Score')
colors = ['#2ecc71' if s <= 1.5 else '#f39c12' if s <= 3 else '#e74c3c' for s in sorted_v['Risk_Score']]
bars = ax.barh(sorted_v['Vendor'], sorted_v['Risk_Score'], color=colors)
ax.set_xlabel('Risk Score (1=Lowest, 5=Highest)', fontsize=12, fontweight='bold')
ax.set_title('Vendor Risk Assessment — InsureLogic AI', fontsize=14, fontweight='bold')
ax.set_xlim(0, 5)
ax.axvline(x=2.5, color='gray', linestyle='--', alpha=0.5, label='Acceptable Threshold')
ax.legend()
for bar in bars:
    w = bar.get_width()
    ax.text(w + 0.1, bar.get_y() + bar.get_height()/2, f'{w:.2f}', ha='left', va='center', fontweight='bold')
plt.tight_layout()
plt.savefig('vendor_risk_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 6: SLA Compliance Dashboard

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

for vendor in vendors_df['Vendor']:
    vd = sla_df[sla_df['Vendor'] == vendor].sort_values('Day')
    ax1.plot(vd['Day'], vd['Uptime'], marker='o', label=vendor, linewidth=1.5, markersize=3)
ax1.axhline(y=uptime_threshold, color='red', linestyle='--', linewidth=2, label=f'SLA Threshold ({uptime_threshold}%)')
ax1.set_ylabel('Uptime (%)', fontweight='bold')
ax1.set_title('30-Day Uptime Tracking', fontsize=13, fontweight='bold')
ax1.legend(loc='lower left', fontsize=9)
ax1.set_ylim(98, 100.2)
ax1.grid(True, alpha=0.3)

for vendor in vendors_df['Vendor']:
    vd = sla_df[sla_df['Vendor'] == vendor].sort_values('Day')
    ax2.plot(vd['Day'], vd['Latency_ms'], marker='o', label=vendor, linewidth=1.5, markersize=3)
ax2.axhline(y=latency_threshold, color='red', linestyle='--', linewidth=2, label=f'SLA Threshold ({latency_threshold}ms)')
ax2.set_xlabel('Day', fontweight='bold')
ax2.set_ylabel('Latency (ms)', fontweight='bold')
ax2.set_title('30-Day Latency Tracking', fontsize=13, fontweight='bold')
ax2.legend(loc='upper left', fontsize=9)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('sla_compliance_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 7: Radar Chart — Multi-Dimensional Comparison

In [ ]:
top3 = vendors_df.nsmallest(3, 'Risk_Score')
categories = ['Security', 'Data Handling', 'Transparency', 'Financial', 'SLA', 'Exit Strategy']
fig, axes = plt.subplots(1, 3, figsize=(18, 6), subplot_kw=dict(projection='polar'))
colors = ['#3498db', '#2ecc71', '#e74c3c']

for idx, (_, row) in enumerate(top3.iterrows()):
    ax = axes[idx]
    values = [row['Security_Score'], row['Data_Handling_Score'], row['Model_Transparency'],
              row['Financial_Stability'], row['SLA_Compliance'], row['Exit_Strategy']]
    values += values[:1]
    angles = [n / len(categories) * 2 * pi for n in range(len(categories))]
    angles += angles[:1]
    ax.plot(angles, values, 'o-', linewidth=2, color=colors[idx])
    ax.fill(angles, values, alpha=0.25, color=colors[idx])
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories, fontsize=9)
    ax.set_ylim(0, 100)
    ax.set_title(f"{row['Vendor']}\n(Risk: {row['Risk_Score']})", fontsize=11, fontweight='bold', pad=20)

plt.suptitle('Multi-Dimensional Vendor Comparison (Top 3)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('vendor_radar.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 8: Final Assessment Report

In [ ]:
print('='*90)
print('VENDOR ASSESSMENT REPORT — InsureLogic AI Claims Processing System')
print('='*90)

for _, row in summary.sort_values('Risk_Score').iterrows():
    print(f"\n  {row['Vendor']}")
    print(f"    Risk Score: {row['Risk_Score']:.2f}/5.0")
    print(f"    Avg Uptime: {row['Avg_Uptime']:.3f}% {'✓' if row['Avg_Uptime'] >= uptime_threshold else '✗'}")
    print(f"    Avg Latency: {row['Avg_Latency']:.1f}ms {'✓' if row['Avg_Latency'] <= latency_threshold else '✗'}")
    print(f"    Recommendation: {row['Recommendation']}")

print('\n' + '='*90)
print(f"Scoring weights used: {scoring_weights}")
print(f"SLA thresholds: Uptime >= {uptime_threshold}%, Latency <= {latency_threshold}ms")

## Step 9: Exit Strategy & Vendor Lock-in Analysis

For GenAI vendors, "exit strategy" extends beyond simple data portability. Consider:

**Exit Strategy Dimensions**:
- **Prompt Template Re-engineering Costs**: How much effort to adapt prompts for competitor's model?
- **Fine-tuning Data Portability**: Can you download fine-tuned weights and adapt to another vendor?
- **Embedding Space Compatibility**: Are embeddings compatible across vendors, or must you re-embed the entire knowledge base?
- **RAG Pipeline Migration**: How easily can you switch LLM providers in your retrieval-augmented generation pipeline?
- **Proprietary Features Lock-in**: Vendor-specific features (function calling, guardrails) may not transfer.

**Strategic Implication**: A vendor with poor exit strategy metrics may look attractive short-term but create long-term switching costs. SafeGuard AI scores well on exit strategy (95) because their APIs follow NovaMind standards, reducing migration costs.

In [ ]:
# Analyze exit strategy dimension
print("\n" + "="*80)
print("EXIT STRATEGY ANALYSIS")
print("="*80)

exit_scores = vendors_df[['Vendor', 'Exit_Strategy']].sort_values('Exit_Strategy', ascending=False)
print("\nExit Strategy Scores (0-100, higher = easier to switch vendors):")
print(exit_scores.to_string(index=False))

print("\nInterpretation:")
print("  Score 90-100: Easy exit — APIs compatible with industry standards")
print("  Score 70-89:  Moderate — Some custom work required to switch")
print("  Score <70:    Difficult — Significant re-engineering costs")

for _, row in exit_scores.iterrows():
    vendor = row['Vendor']
    score = row['Exit_Strategy']
    if score >= 90:
        migration_cost = "Low"
    elif score >= 70:
        migration_cost = "Moderate"
    else:
        migration_cost = "High"
    
    print(f"\n  {vendor}: {score}/100 — {migration_cost} switching cost")

## Step 10: Sensitivity Analysis — Score Confidence & Tier Stability

Small changes in vendor scores can flip recommendation tiers. This analysis tests how robust our recommendations are.

In [ ]:
# Sensitivity analysis: adjust one vendor's scores ±10 points
print("\n" + "="*80)
print("SENSITIVITY ANALYSIS: Impact of ±10 Point Score Adjustments")
print("="*80)

def recalculate_with_adjustment(vendor_name, adjustment_percent=10):
    """Recalculate risk score for a vendor with adjusted evaluation scores"""
    df_adjusted = vendors_df.copy()
    
    # Find vendor row
    vendor_idx = df_adjusted[df_adjusted['Vendor'] == vendor_name].index[0]
    
    # Adjust all criterion scores by ±adjustment_percent
    for col in ['Security_Score', 'Data_Handling_Score', 'Model_Transparency', 
                'Financial_Stability', 'SLA_Compliance', 'Exit_Strategy', 'AI_Model_Governance']:
        current = df_adjusted.loc[vendor_idx, col]
        # Adjust by percentage, capped at 0-100
        adjusted = max(0, min(100, current + adjustment_percent))
        df_adjusted.loc[vendor_idx, col] = adjusted
    
    # Recalculate risk
    new_risk = calculate_risk_scores(df_adjusted[df_adjusted.index == vendor_idx], scoring_weights)[0]
    return df_adjusted.loc[vendor_idx], new_risk

print("\nTesting: What if we increase/decrease vendor scores by 10 points?\n")

for vendor in vendors_df['Vendor']:
    original_risk = vendors_df[vendors_df['Vendor'] == vendor]['Risk_Score'].iloc[0]
    original_rec = summary[summary['Vendor'] == vendor]['Recommendation'].iloc[0]
    
    _, risk_minus = recalculate_with_adjustment(vendor, adjustment_percent=-10)
    _, risk_plus = recalculate_with_adjustment(vendor, adjustment_percent=+10)
    
    # Determine recommendations for adjusted scores
    sla_compliant = summary[summary['Vendor'] == vendor]['SLA_Compliant'].iloc[0]
    
    def rec(score, sla):
        if score <= 1.5 and sla:
            return 'Highly Recommended'
        elif score <= 2.5 and sla:
            return 'Recommended'
        elif score <= 3.5:
            return 'Acceptable (with monitoring)'
        else:
            return 'Not Recommended'
    
    rec_minus = rec(risk_minus, sla_compliant)
    rec_plus = rec(risk_plus, sla_compliant)
    
    print(f"{vendor}:")
    print(f"  Baseline:        Risk {original_risk:.2f} → {original_rec}")
    print(f"  Scores -10 pts:  Risk {risk_minus:.2f} → {rec_minus}")
    print(f"  Scores +10 pts:  Risk {risk_plus:.2f} → {rec_plus}")
    
    # Highlight if tiers change
    if original_rec != rec_minus or original_rec != rec_plus:
        print(f"  ⚠️  TIER CHANGE: Small score adjustments flip recommendation tier!")
    print()

print("\nConclusion:")
print("Vendor recommendations are sensitive to scoring inputs. Strong governance requires:")
print("  - Rigorous scoring methodology with clear rubrics")
print("  - Regular re-evaluation (quarterly minimum)")
print("  - Confidence intervals around scores")
print("  - Cross-functional review to validate assumptions")